# YOLOv8m-OBB — Intel Arc A750 (Updated)

This version is adjusted for your current setup:

- Windows 11
- Python 3.11
- Intel Arc A750 8 GB
- PyTorch XPU
- YOLOv8m-OBB
- 80% train / 10% validation / 10% test
- 20 epochs
- Per-epoch Precision, Recall, mAP50, and mAP50-95
- Automatic removal of invalid image/label pairs
- `AMP=False` to avoid the Intel XPU `fp64` GradScaler error
- Conservative `BATCH=2` and `WORKERS=0` for stability on Windows + 8 GB VRAM

## Important

You already installed the correct XPU environment.  
**There is no uninstall/reinstall cell in this notebook.**

If you ever need to install the environment again, do it from PowerShell, not by repeatedly running notebook install cells.


## 1. Verify that VS Code is using the correct kernel

Select the project environment:

`...\yolov8-obb\.venv\Scripts\python.exe`

Then run this cell. It must show:

- PyTorch version ending in `+xpu`
- `XPU available: True`
- Intel Arc A750


In [ ]:
import sys
import torch
import torchvision
import ultralytics

print("Python executable:")
print(sys.executable)

print("\nPyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Ultralytics:", ultralytics.__version__)

print("\nXPU exists:", hasattr(torch, "xpu"))
print("XPU available:", torch.xpu.is_available() if hasattr(torch, "xpu") else False)

if not hasattr(torch, "xpu") or not torch.xpu.is_available():
    raise RuntimeError(
        "Intel XPU is not available. Select the project's .venv kernel before continuing."
    )

print("GPU:", torch.xpu.get_device_name(0))

device_test = torch.device("xpu:0")
a = torch.randn((512, 512), device=device_test)
b = torch.randn((512, 512), device=device_test)
c = a @ b
print("XPU calculation test:", c.device, "OK")

del a, b, c
torch.xpu.empty_cache()


## 2. Configuration

Your project is expected to look like:

```text
D:\path\to\yolov8-obb\
├── images\
├── labels\
├── .venv\
└── this_notebook.ipynb
```

For one license-plate class, keep:

```python
CLASS_NAMES = ["license_plate"]
```


In [ ]:
from pathlib import Path
import random
import shutil
import yaml
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(r"D:\academy\thesis\dataset\thesis dataset\yolov8m-obb\input")  # TODO: set your local project path

SOURCE_IMAGES = PROJECT_ROOT / "images"
SOURCE_LABELS = PROJECT_ROOT / "labels"

CLASS_NAMES = ["license_plate"]

SEED = 42
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

EPOCHS = 20
IMG_SIZE = 640

# Conservative values for Intel Arc A750 8 GB + Windows
BATCH = 2
WORKERS = 0

# IMPORTANT: keep AMP disabled for this XPU setup.
AMP = False

DEVICE = "xpu:0"
RUN_NAME = "yolov8m_obb_20ep_xpu"

SPLIT_ROOT = PROJECT_ROOT / "dataset_80_10_10"
INVALID_ROOT = PROJECT_ROOT / "removed_invalid_samples"

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"
}

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

print("Project root:", PROJECT_ROOT)
print("Images:", SOURCE_IMAGES)
print("Labels:", SOURCE_LABELS)
print("Device:", DEVICE)
print("AMP:", AMP)
print("Batch:", BATCH)
print("Workers:", WORKERS)
print("Epochs:", EPOCHS)


## 3. Scan dataset and move invalid image/label pairs out

This cell checks all image/label pairs.

A valid YOLOv8-OBB row should contain:

```text
class_id x1 y1 x2 y2 x3 y3 x4 y4
```

All 8 coordinates must be between `0` and `1`.

If an annotation is invalid, **both its image and label are moved** to:

```text
removed_invalid_samples/
├── images/
└── labels/
```

They are not permanently deleted.

Empty `.txt` files are allowed and are kept as background/negative images.


In [ ]:
assert SOURCE_IMAGES.exists(), f"Images folder not found: {SOURCE_IMAGES}"
assert SOURCE_LABELS.exists(), f"Labels folder not found: {SOURCE_LABELS}"
assert len(CLASS_NAMES) > 0, "CLASS_NAMES cannot be empty."

INVALID_IMAGES = INVALID_ROOT / "images"
INVALID_LABELS = INVALID_ROOT / "labels"
INVALID_IMAGES.mkdir(parents=True, exist_ok=True)
INVALID_LABELS.mkdir(parents=True, exist_ok=True)

def get_image_files():
    return sorted(
        p for p in SOURCE_IMAGES.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

def validate_label_file(label_path):
    errors = []
    try:
        raw_text = label_path.read_text(encoding="utf-8-sig")
    except Exception as e:
        return [f"Could not read file: {e}"]

    lines = [line.strip() for line in raw_text.splitlines() if line.strip()]

    for line_no, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) != 9:
            errors.append(f"line {line_no}: expected 9 values, got {len(parts)}")
            continue

        try:
            class_value = float(parts[0])
            coords = [float(v) for v in parts[1:]]
        except ValueError:
            errors.append(f"line {line_no}: contains non-numeric value(s)")
            continue

        if not class_value.is_integer():
            errors.append(f"line {line_no}: class ID is not an integer")
            continue

        class_id = int(class_value)
        if not (0 <= class_id < len(CLASS_NAMES)):
            errors.append(
                f"line {line_no}: class ID {class_id} is outside 0..{len(CLASS_NAMES)-1}"
            )

        if any(v < 0.0 or v > 1.0 for v in coords):
            errors.append(
                f"line {line_no}: one or more coordinates are outside 0..1 "
                f"(min={min(coords):.6f}, max={max(coords):.6f})"
            )

    return errors

image_files = get_image_files()
if not image_files:
    raise RuntimeError(f"No supported images found in: {SOURCE_IMAGES}")

stem_to_images = {}
for p in image_files:
    stem_to_images.setdefault(p.stem, []).append(p)

duplicate_stems = {stem: paths for stem, paths in stem_to_images.items() if len(paths) > 1}
if duplicate_stems:
    examples = {stem: [p.name for p in paths] for stem, paths in list(duplicate_stems.items())[:10]}
    raise RuntimeError("Duplicate image stems found. Fix these manually first:\n" + str(examples))

missing_labels = [img for img in image_files if not (SOURCE_LABELS / f"{img.stem}.txt").exists()]
if missing_labels:
    print("Images missing labels:")
    for p in missing_labels[:20]:
        print(" -", p.name)
    raise RuntimeError(
        f"{len(missing_labels)} image(s) have no matching TXT label. Fix or remove those pairs before continuing."
    )

all_labels = sorted(p for p in SOURCE_LABELS.iterdir() if p.is_file() and p.suffix.lower() == ".txt")
image_stems = {p.stem for p in image_files}
orphan_labels = [p for p in all_labels if p.stem not in image_stems]

if orphan_labels:
    print("WARNING: label files without matching images will be ignored:")
    for p in orphan_labels[:20]:
        print(" -", p.name)

invalid_pairs = []
for img_path in image_files:
    label_path = SOURCE_LABELS / f"{img_path.stem}.txt"
    errors = validate_label_file(label_path)
    if errors:
        invalid_pairs.append((img_path, label_path, errors))

print(f"Images before cleaning: {len(image_files)}")
print(f"Invalid image/label pairs found: {len(invalid_pairs)}")

for img_path, label_path, errors in invalid_pairs:
    print("\nINVALID:", label_path.name)
    for e in errors:
        print("  -", e)

    dst_img = INVALID_IMAGES / img_path.name
    dst_lbl = INVALID_LABELS / label_path.name

    if img_path.exists():
        shutil.move(str(img_path), str(dst_img))
    if label_path.exists():
        shutil.move(str(label_path), str(dst_lbl))

    print("  moved image ->", dst_img)
    print("  moved label ->", dst_lbl)

print("\nCleaning complete.")
print("Invalid samples folder:", INVALID_ROOT)


## 4. Re-check the cleaned dataset

This must end with:

`DATASET CHECK PASSED.`


In [ ]:
image_files = get_image_files()
all_labels = sorted(p for p in SOURCE_LABELS.iterdir() if p.is_file() and p.suffix.lower() == ".txt")

missing_labels = [p for p in image_files if not (SOURCE_LABELS / f"{p.stem}.txt").exists()]
image_stems = {p.stem for p in image_files}
orphan_labels = [p for p in all_labels if p.stem not in image_stems]

format_errors = []
object_count = 0
empty_label_count = 0
class_ids_seen = set()

for img_path in image_files:
    label_path = SOURCE_LABELS / f"{img_path.stem}.txt"
    errors = validate_label_file(label_path)
    if errors:
        format_errors.append((label_path.name, errors))
        continue

    lines = [
        line.strip()
        for line in label_path.read_text(encoding="utf-8-sig").splitlines()
        if line.strip()
    ]

    if not lines:
        empty_label_count += 1
        continue

    for line in lines:
        parts = line.split()
        class_id = int(float(parts[0]))
        class_ids_seen.add(class_id)
        object_count += 1

print("Images found:", len(image_files))
print("TXT labels found:", len(all_labels))
print("Images missing labels:", len(missing_labels))
print("Labels without matching images:", len(orphan_labels))
print()
print("Objects found in labels:", object_count)
print("Empty/background label files:", empty_label_count)
print("Class IDs found:", sorted(class_ids_seen))

if missing_labels:
    raise RuntimeError("Some images are still missing labels.")

if format_errors:
    for name, errs in format_errors[:20]:
        print("\n", name)
        for e in errs:
            print(" -", e)
    raise RuntimeError("Invalid OBB labels still remain.")

print("\nDATASET CHECK PASSED.")


## 5. Create a fresh random 80/10/10 split

The original cleaned `images/` and `labels/` folders remain intact.

The generated dataset will be:

```text
dataset_80_10_10/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
├── labels/
│   ├── train/
│   ├── val/
│   └── test/
└── data.yaml
```

The fixed seed makes the split reproducible.


In [ ]:
if SPLIT_ROOT.exists():
    shutil.rmtree(SPLIT_ROOT)

for split in ("train", "val", "test"):
    (SPLIT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (SPLIT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

items = list(get_image_files())
rng = random.Random(SEED)
rng.shuffle(items)

n = len(items)
n_train = round(n * TRAIN_RATIO)
n_val = round(n * VAL_RATIO)
n_test = n - n_train - n_val

if min(n_train, n_val, n_test) < 1:
    raise RuntimeError(
        f"Dataset too small for 80/10/10 split: train={n_train}, val={n_val}, test={n_test}"
    )

split_items = {
    "train": items[:n_train],
    "val": items[n_train:n_train + n_val],
    "test": items[n_train + n_val:]
}

for split, split_images in split_items.items():
    for img_path in split_images:
        label_path = SOURCE_LABELS / f"{img_path.stem}.txt"
        shutil.copy2(img_path, SPLIT_ROOT / "images" / split / img_path.name)
        shutil.copy2(label_path, SPLIT_ROOT / "labels" / split / label_path.name)

print("Split complete:")
for split in ("train", "val", "test"):
    count = len(split_items[split])
    print(f"{split:5s}: {count:4d} images ({100 * count / n:.2f}%)")
print("total:", n)


## 6. Create `data.yaml`


In [ ]:
DATA_YAML = SPLIT_ROOT / "data.yaml"

yaml_data = {
    "path": SPLIT_ROOT.resolve().as_posix(),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
}

with DATA_YAML.open("w", encoding="utf-8") as f:
    yaml.safe_dump(yaml_data, f, sort_keys=False, allow_unicode=True)

print(DATA_YAML.read_text(encoding="utf-8"))
print("Created:", DATA_YAML)


## 7. Smoke test

This checks that:

- Ultralytics can load YOLOv8m-OBB
- the generated dataset can be read
- validation can run on `xpu:0`

It does **not** train for 20 epochs yet.


In [ ]:
from ultralytics import YOLO
import torch

print("Device:", DEVICE)
print("AMP:", AMP)

if not torch.xpu.is_available():
    raise RuntimeError("XPU became unavailable. Restart the correct .venv kernel.")

torch.xpu.empty_cache()

model = YOLO("yolov8m-obb.pt")

smoke_metrics = model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    plots=False,
    verbose=True
)

print("\nSMOKE TEST PASSED.")
print(f"Initial Precision   : {float(smoke_metrics.box.mp) * 100:.2f}%")
print(f"Initial Recall      : {float(smoke_metrics.box.mr) * 100:.2f}%")
print(f"Initial mAP50       : {float(smoke_metrics.box.map50) * 100:.2f}%")
print(f"Initial mAP50-95    : {float(smoke_metrics.box.map) * 100:.2f}%")

del model
torch.xpu.empty_cache()


## 8. Train YOLOv8m-OBB for 20 epochs

### Important Intel Arc setting

This training intentionally uses:

```python
device="xpu:0"
amp=False
```

`amp=False` avoids the XPU GradScaler error:

```text
RuntimeError: Required aspect fp64 is not supported on the device
```

Your Intel Arc GPU is still used; only AMP/mixed precision is disabled.

The callback prints validation metrics after every epoch.


In [ ]:
from ultralytics import YOLO
import torch

print("Training configuration")
print("----------------------")
print("Device :", DEVICE)
print("AMP    :", AMP)
print("Epochs :", EPOCHS)
print("Batch  :", BATCH)
print("Workers:", WORKERS)
print("Image  :", IMG_SIZE)

assert DEVICE == "xpu:0"
assert AMP is False
assert torch.xpu.is_available()

torch.xpu.empty_cache()

model = YOLO("yolov8m-obb.pt")

def print_epoch_metrics(trainer):
    metrics = getattr(trainer, "metrics", {}) or {}
    epoch = int(getattr(trainer, "epoch", 0)) + 1

    p = metrics.get("metrics/precision(B)")
    r = metrics.get("metrics/recall(B)")
    map50 = metrics.get("metrics/mAP50(B)")
    map5095 = metrics.get("metrics/mAP50-95(B)")

    if any(v is not None for v in (p, r, map50, map5095)):
        def pct(value):
            return "N/A" if value is None else f"{float(value) * 100:.2f}%"

        print(
            f"\n>>> Epoch {epoch:02d}/{EPOCHS} | "
            f"Precision={pct(p)} | "
            f"Recall={pct(r)} | "
            f"mAP50={pct(map50)} | "
            f"mAP50-95={pct(map5095)}"
        )

model.add_callback("on_fit_epoch_end", print_epoch_metrics)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    amp=False,
    pretrained=True,
    val=True,
    project=str(PROJECT_ROOT / "runs_obb"),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    verbose=True,
    seed=SEED,
    deterministic=False,
    cache=False
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_MODEL = RUN_DIR / "weights" / "best.pt"
LAST_MODEL = RUN_DIR / "weights" / "last.pt"

print("\nTRAINING FINISHED.")
print("Run folder :", RUN_DIR)
print("Best model :", BEST_MODEL)
print("Last model :", LAST_MODEL)


### If you get an out-of-memory error

This notebook already uses the conservative setting:

```python
BATCH = 2
```

If 2 is still too large, change it to:

```python
BATCH = 1
```

Restart the kernel, rerun the configuration and required preparation cells, then start training again.

Do **not** turn AMP back on for this current XPU environment.


## 9. Show validation metrics for every epoch

Object detection does not normally use one classification-style `accuracy` value.

The useful performance metrics are:

- Precision
- Recall
- mAP@0.5
- mAP@0.5:0.95


In [ ]:
RESULTS_CSV = RUN_DIR / "results.csv"
assert RESULTS_CSV.exists(), f"results.csv not found: {RESULTS_CSV}"

df = pd.read_csv(RESULTS_CSV)
df.columns = [c.strip() for c in df.columns]

wanted = [
    "epoch",
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)"
]

available = [c for c in wanted if c in df.columns]
epoch_metrics = df[available].copy()

if "epoch" in epoch_metrics.columns:
    epoch_metrics["epoch"] = epoch_metrics["epoch"].astype(int) + 1

epoch_metrics = epoch_metrics.rename(columns={
    "epoch": "Epoch",
    "metrics/precision(B)": "Precision (%)",
    "metrics/recall(B)": "Recall (%)",
    "metrics/mAP50(B)": "mAP50 (%)",
    "metrics/mAP50-95(B)": "mAP50-95 (%)"
})

for c in ["Precision (%)", "Recall (%)", "mAP50 (%)", "mAP50-95 (%)"]:
    if c in epoch_metrics.columns:
        epoch_metrics[c] = epoch_metrics[c].astype(float) * 100

display(epoch_metrics.round(2))

SUMMARY_CSV = RUN_DIR / "epoch_metrics_percent.csv"
epoch_metrics.to_csv(SUMMARY_CSV, index=False)
print("\nSaved:", SUMMARY_CSV)


In [ ]:
plt.figure(figsize=(9, 5))

if "mAP50 (%)" in epoch_metrics.columns:
    plt.plot(epoch_metrics["Epoch"], epoch_metrics["mAP50 (%)"], marker="o", label="mAP50")

if "mAP50-95 (%)" in epoch_metrics.columns:
    plt.plot(epoch_metrics["Epoch"], epoch_metrics["mAP50-95 (%)"], marker="o", label="mAP50-95")

plt.xlabel("Epoch")
plt.ylabel("Metric (%)")
plt.title("YOLOv8m-OBB Validation mAP by Epoch")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 10. Final evaluation on the untouched 10% test set

This loads `best.pt` and evaluates only the test split.


In [ ]:
from ultralytics import YOLO
import torch

assert BEST_MODEL.exists(), f"best.pt not found: {BEST_MODEL}"

torch.xpu.empty_cache()
best_model = YOLO(str(BEST_MODEL))

test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    plots=True,
    verbose=True
)

print("\n===== FINAL TEST-SET METRICS =====")
print(f"Precision    : {float(test_metrics.box.mp) * 100:.2f}%")
print(f"Recall       : {float(test_metrics.box.mr) * 100:.2f}%")
print(f"mAP@0.5      : {float(test_metrics.box.map50) * 100:.2f}%")
print(f"mAP@0.5:0.95 : {float(test_metrics.box.map) * 100:.2f}%")
print("==================================")


## 11. Optional: save OBB predictions for test images


In [ ]:
PREDICT_SOURCE = SPLIT_ROOT / "images" / "test"

prediction_results = best_model.predict(
    source=str(PREDICT_SOURCE),
    imgsz=IMG_SIZE,
    conf=0.25,
    device=DEVICE,
    save=True,
    project=str(PROJECT_ROOT / "runs_obb"),
    name=f"{RUN_NAME}_test_predictions",
    exist_ok=True
)

print("Saved predictions to:")
print(PROJECT_ROOT / "runs_obb" / f"{RUN_NAME}_test_predictions")


# Important output files

After successful training:

```text
runs_obb/
└── yolov8m_obb_20ep_xpu/
    ├── results.csv
    ├── epoch_metrics_percent.csv
    ├── results.png
    └── weights/
        ├── best.pt
        └── last.pt
```

Use **`best.pt`** as your final trained model.

## Recommended run order after a kernel restart

1. Kernel/XPU verification
2. Configuration
3. Dataset cleaning
4. Dataset re-check
5. Split
6. `data.yaml`
7. Smoke test
8. 20-epoch training
9. Per-epoch metrics
10. Final test evaluation

Do not rerun any PyTorch uninstall/install command from inside this notebook.
